# Phase 0 smoke test: GRPO forecasting on a free Kaggle GPU

This notebook checks that the whole training setup works before we touch real Kalshi data. It trains **Llama 3.2 3B Instruct** with GRPO for 20 steps on a toy forecasting task that uses the project's real output format (`Probability: 0.XX`) and reward (a Brier-score reward).

**Before you press Run All**, in the right-hand sidebar under *Settings*:

1. **Accelerator → GPU T4 x2** (needs a phone-verified Kaggle account)
2. **Internet → On**

Expect about 25–40 minutes: roughly 5–10 to install, 15–25 to train, a few to evaluate.

Adapted from Unsloth's Llama 3.2 (3B) GRPO notebook (LGPL-3.0).

## 1. Settings

The only cell you should need to edit. On the **first run**, leave `RESUME_FROM = None`. The resume test at the bottom explains the second run.

In [ ]:
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"  # ungated copy of Meta's model; knowledge cutoff Dec 2023
MAX_STEPS = 20        # smoke test length. Set to 25 for the resume test.
SAVE_STEPS = 10       # writes a checkpoint every 10 steps
RESUME_FROM = None    # second run only: set to "auto" to find the previous run's checkpoints in your inputs
OUTPUT_DIR = "/kaggle/working/outputs"
LORA_DIR = "/kaggle/working/lora_final"
SEED = 3407

## 2. Check the GPU

Unsloth trains on one GPU, so we pin it to the first of Kaggle's two T4s. If this cell fails, the accelerator is off or your phone number isn't verified.

In [ ]:
import os, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # must run before torch is imported

try:
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"]).decode())
except Exception as e:
    raise RuntimeError(
        "No GPU found. In the right sidebar set Accelerator to 'GPU T4 x2', "
        "and check that your phone number is verified in Kaggle Settings."
    ) from e

## 3. Install Unsloth, vLLM and TRL

Same pinned versions as Unsloth's own Kaggle notebook. Takes 5–10 minutes. Warnings are normal; a red error that stops the cell is not.

In [ ]:
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
!pip install --upgrade -qqq uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
_vllm, _triton = ('vllm==0.11.2', 'triton') if is_t4 else ('vllm==0.15.1', 'triton')
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton} "huggingface_hub>=0.34.0" "datasets==4.3.0"
try:
    import importlib.metadata as _md; _torch_v = tuple(int(_p) for _p in _md.version("torch").split("+")[0].split(".")[:2])
except Exception:
    _torch_v = ()
_torchao = "torchao>=0.16.0" if _torch_v >= (2, 10) else "torchao>=0.16.0,<0.18.0"
!uv pip install -qqq --no-deps --upgrade "{_torchao}"
!uv pip install transformers==4.57.6
!uv pip install --no-deps trl==0.22.2

## 4. Load the model with LoRA

`gpu_memory_utilization` is the share of GPU memory vLLM reserves for fast generation. If you hit an out-of-memory error, lower it to 0.7.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024  # our prompts and answers are short
lora_rank = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    load_in_4bit = False,          # 16-bit LoRA, as in Unsloth's notebook
    fast_inference = True,         # vLLM for fast sampling during RL
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9,  # lower to 0.7 if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

## 5. Toy forecasting task

Each question describes an urn with some red and blue balls and asks whether a randomly drawn ball will be red. The correct probability is the share of red balls, but **the reward only sees what actually happened** (one simulated draw), exactly like a resolved market. That makes this a small version of the real project: the model can only learn good probabilities by being rewarded on noisy yes/no outcomes.

`true_p` (the real share of red balls) is stored only so we can check calibration afterwards; the model never sees it.

In [ ]:
import random, re
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are a careful forecaster. Think briefly (a few sentences), "
    "then finish with one final line in exactly this form:\n"
    "Probability: 0.XX\n"
    "where 0.XX is your probability that the answer is YES."
)

def make_toy_questions(n, seed):
    rng = random.Random(seed)
    rows = []
    for _ in range(n):
        red, blue = rng.randint(1, 20), rng.randint(1, 20)
        p = red / (red + blue)
        rows.append({
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": (
                    f"An urn holds {red} red balls and {blue} blue balls. "
                    "One ball is drawn at random. Will it be red?"
                )},
            ],
            "outcome": int(rng.random() < p),  # what "happened": 1 = red, 0 = blue
            "true_p": p,                       # for checking calibration only
        })
    return rows

train_rows = make_toy_questions(400, SEED)
test_rows = make_toy_questions(50, SEED + 1)
dataset = Dataset.from_list(train_rows)
print(train_rows[0]["prompt"][-1]["content"], "| outcome:", train_rows[0]["outcome"])

## 6. Parser and rewards

Two rewards are added together:

- **Format reward:** +0.5 if the answer ends with a readable `Probability: 0.XX` between 0 and 1, otherwise −1.0.
- **Brier reward:** 1 minus the squared gap between the stated probability and what happened (1 if red, 0 if blue). A perfect, confident forecast scores 1; saying 0.5 always scores 0.75; a confident wrong forecast scores 0. Because the Brier score is a proper scoring rule, the best long-run strategy is to state your true belief.

Every 5th reward call prints one sample so you can read what the model is writing.

In [ ]:
PROB_LINE = re.compile(r"Probability:([^\n]*)", re.IGNORECASE)
NUMBER = re.compile(r"\d*\.?\d+")

def extract_probability(text):
    """Takes the last 'Probability:' line and returns the last number on it, if it is in [0, 1].
    So 'Probability: 8/17 = 0.47' gives 0.47. Percentages are rejected: the prompt asks for 0.XX."""
    lines = PROB_LINE.findall(text)
    if not lines:
        return None
    line = lines[-1]
    if "%" in line:
        return None
    numbers = NUMBER.findall(line)
    if not numbers:
        return None
    p = float(numbers[-1])
    return p if 0.0 <= p <= 1.0 else None

def format_reward(completions, **kwargs):
    return [0.5 if extract_probability(c[0]["content"]) is not None else -1.0 for c in completions]

_reward_calls = 0
def brier_reward(prompts, completions, outcome, **kwargs):
    global _reward_calls
    scores = []
    for c, y in zip(completions, outcome):
        p = extract_probability(c[0]["content"])
        scores.append(0.0 if p is None else 1.0 - (p - y) ** 2)
    if _reward_calls % 5 == 0:
        print("=" * 70)
        print(prompts[0][-1]["content"], "| outcome:", outcome[0])
        print("-" * 70)
        print(completions[0][0]["content"][:600])
        print("-" * 70, "\nreward:", round(scores[0], 3))
    _reward_calls += 1
    return scores

Quick check that the parser and rewards behave as described (runs instantly):

In [ ]:
def _c(text):
    return [{"content": text}]

assert extract_probability("Some reasoning.\nProbability: 0.35") == 0.35
assert extract_probability("Probability: 0.2 ... actually Probability: 0.4") == 0.4  # last one wins
assert extract_probability("Probability: 8/17 = 0.47") == 0.47                        # fraction then decimal
assert extract_probability("Probability: 0.3\nOn reflection,\nProbability: 0.25") == 0.25
assert extract_probability("Probability: 70%") is None                                # must be a decimal
assert extract_probability("Probability: 1.5") is None                                # out of range
assert extract_probability("no number here") is None
assert format_reward([_c("Probability: 0.6"), _c("I think 60%")]) == [0.5, -1.0]
_saved = _reward_calls
_reward_calls = 1  # skip the sample printout for this test
r = brier_reward([[{"content": "q"}]] * 3, [_c("Probability: 1.0"), _c("Probability: 0.5"), _c("oops")], [1, 1, 1])
_reward_calls = _saved
assert [round(x, 4) for x in r] == [1.0, 0.75, 0.0]
print("Parser and rewards OK")

## 7. Train with GRPO

For each question the model writes 4 answers, the rewards are compared within that group, and the model is nudged toward the better ones. Checkpoints are saved every 10 steps to `/kaggle/working/outputs`.

In the log table, `reward` should mostly stay between about 1.0 and 1.5, and `completion_length` should stay well under 256. Twenty steps is too short for big gains; this run checks that everything works.

In [ ]:
import glob, os, shutil
from trl import GRPOConfig, GRPOTrainer

max_prompt_length = max(
    len(tokenizer.apply_chat_template(r["prompt"], add_generation_prompt = True, tokenize = True))
    for r in train_rows
) + 8

training_args = GRPOConfig(
    learning_rate = 5e-6,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations = 4,
    max_prompt_length = max_prompt_length,
    max_completion_length = 256,
    max_steps = MAX_STEPS,
    save_steps = SAVE_STEPS,
    max_grad_norm = 1.0,
    report_to = "none",
    output_dir = OUTPUT_DIR,
    seed = SEED,
)

resume = False
if RESUME_FROM == "auto":
    _ckpts = glob.glob("/kaggle/input/**/checkpoint-*", recursive = True)
    if not _ckpts:
        raise FileNotFoundError("No checkpoints in /kaggle/input. Add the previous version as an input (Add Input > Your Work > Notebooks).")
    _latest = max(_ckpts, key = lambda p: int(p.rsplit("-", 1)[-1]))
    RESUME_FROM = os.path.dirname(_latest)
    print("Found checkpoints in", RESUME_FROM)
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, OUTPUT_DIR, dirs_exist_ok = True)
    print("Resuming. Checkpoints found:", sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*")))
    resume = True

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [format_reward, brier_reward],
    args = training_args,
    train_dataset = dataset,
)
trainer.train(resume_from_checkpoint = resume)

The live training table doesn't show up in a saved version's logs, so this prints the same numbers as plain text. After a resume, the first step listed should be 21.

In [ ]:
import pandas as pd

wanted = ["step", "reward", "reward_std", "completion_length", "kl",
          "rewards/format_reward/mean", "rewards/brier_reward/mean"]
history = pd.DataFrame([h for h in trainer.state.log_history if "reward" in h])
cols = [c for c in wanted if c in history.columns]
print(history[cols].round(4).to_string(index = False))

## 8. Save the trained LoRA and confirm it changed

In [ ]:
from safetensors import safe_open

model.save_lora(LORA_DIR)
with safe_open(f"{LORA_DIR}/adapter_model.safetensors", framework = "pt") as f:
    for key in f.keys():
        t = f.get_tensor(key)
        assert (t == 0).sum().item() != t.numel(), f"{key} is all zeros"
print("Saved LoRA to", LORA_DIR, "and its weights are non-zero")

## 9. Compare the base model with the trained one

On 50 held-out urn questions, one answer each:

- **format_ok:** share of answers with a readable probability
- **brier:** average squared gap between the probability and the outcome (lower is better)
- **gap_to_true_p:** average absolute gap between the stated probability and the real share of red balls (lower is better; only possible in this toy task)

After only 20 steps, the two may look similar. That's fine; the goal here is that every step runs.

In [ ]:
from vllm import SamplingParams

sampling_params = SamplingParams(temperature = 0.7, top_p = 0.95, max_tokens = 256, seed = SEED)
eval_texts = [tokenizer.apply_chat_template(r["prompt"], tokenize = False, add_generation_prompt = True) for r in test_rows]

def evaluate(lora_request):
    outs = model.fast_generate(eval_texts, sampling_params = sampling_params, lora_request = lora_request)
    probs = [extract_probability(o.outputs[0].text) for o in outs]
    valid = [(p, r) for p, r in zip(probs, test_rows) if p is not None]
    if not valid:
        return {"format_ok": 0.0}
    return {
        "format_ok": round(len(valid) / len(test_rows), 3),
        "brier": round(sum((p - r["outcome"]) ** 2 for p, r in valid) / len(valid), 4),
        "gap_to_true_p": round(sum(abs(p - r["true_p"]) for p, r in valid) / len(valid), 4),
    }

print("Base model   :", evaluate(None))
print("After GRPO   :", evaluate(model.load_lora(LORA_DIR)))

## 10. Resume test (Phase 0 exit check)

Kaggle sessions end after 9–12 hours, so real training runs will need to stop and pick up again. To test that:

1. **First run:** click **Save Version → Save & Run All (Commit)** at the top right. Only saved versions keep their output files; files from an interactive session disappear when it ends. Wait for the version to finish.
2. **Second run:** open this notebook again, click **Add Input** in the right sidebar, choose **Your Work → Notebooks**, and add this notebook's finished version.
3. In the Settings cell, set `RESUME_FROM = "auto"` and `MAX_STEPS = 25`. The notebook finds the checkpoints in your inputs by itself (the cell below shows where they are, if you're curious).
4. Save Version → Save & Run All. **Pass** = the training log printed under step 7 starts at step 21 and ends at step 25.

You can do the second run in a brand-new notebook made from an updated copy of this file: the checkpoint is copied in from the input, so it doesn't have to be the same notebook.

In [ ]:
import glob, os
found = sorted(glob.glob("/kaggle/input/**/checkpoint-*", recursive = True))
if found:
    print("Checkpoints in your inputs:")
    for p in found:
        print("  ", p)
    print("\nSet RESUME_FROM to the folder that contains them:", os.path.dirname(found[-1]))
else:
    print("No checkpoints in /kaggle/input yet. Add the previous version as an input first (steps 1-2 above).")